# 2. Occupation-task profiles → relatedness

Calculate RCA from the entire USAJOBS corpus, then work through Registered Nurses (29-1141.00) and Medical and Health Services Managers (11-9111.00). Computing the reference from only these two occupations would change the model.

This notebook runs from Notebook 1's counts if present, otherwise from the identical aggregate inputs in `data/`. It uses no private NLx records and no neural model downloads. Install `requirements.txt` first.

**Model:** task-incidence share within an occupation divided by pooled task-incidence share; square root of RCA; normalize so squared weights sum to one; multiply the two occupations' weights and sum across shared tasks. These are embedding-independent profile similarities, not TitleMatch confidence scores.

In [1]:
from pathlib import Path
import os, json, hashlib
import numpy as np
import pandas as pd
from IPython.display import display
DATA = Path("data")
OUT = Path("results")
OUT.mkdir(exist_ok=True)
pd.set_option("display.max_colwidth", 100)

In [2]:
from scipy import sparse
SOURCE = OUT if all((OUT/f).exists() for f in ['incidence8.parquet','totals8.parquet']) else DATA
print('Counts read from:',SOURCE)
inc = pd.read_parquet(SOURCE/'incidence8.parquet')
totals = pd.read_parquet(SOURCE/'totals8.parquet')
tasks = pd.read_csv(DATA/'tasks.csv')
titles = pd.read_csv(DATA/'occupations.csv').set_index('O*NET-SOC Code').Title
codes = sorted(totals.loc[totals.code.isin(titles.index),'code'])
task_ids = sorted(tasks.task_id.unique())
ci={c:i for i,c in enumerate(codes)}; ti={t:i for i,t in enumerate(task_ids)}
inc=inc[inc.code.isin(codes)&inc.task_id.isin(task_ids)].copy()
assert not inc.duplicated(['code','task_id']).any()
assert np.isfinite(inc.weight).all() and (inc.weight>0).all()
print(len(codes),'reference occupations;',len(inc),'valid links before support filtering')

Counts read from: results


994 reference occupations; 896071 valid links before support filtering


## Declare the RCA reference

The default pools retained task incidences after the paper’s pair-level exclusions and minimum support of five postings. Unreviewed links remain. The exclusion file is supplied with the public inputs; this transfers the existing decisions rather than performing a new USAJOBS review. Set `REFERENCE_STAGE="before_support"` only to reproduce the earlier USAJOBS specification. This change affects both reference shares and scores. We retain RCA values below one. We do not treat posting prevalence as task-incidence share.

In [3]:
# Apply the same pair-level exclusions as the NLx production model.
# Unreviewed links remain; RCA is recomputed after exclusions.
exclusions=pd.read_csv(DATA/'occupation_task_exclusions.csv')
assert not exclusions.duplicated(['code','task_id']).any()
inc=inc.merge(exclusions[['code','task_id']].assign(exclude=True),on=['code','task_id'],how='left',validate='one_to_one')
print('Excluded links:',int(inc.exclude.eq(True).sum()))
inc=inc.loc[~inc.exclude.eq(True)].drop(columns='exclude')
SUPPORT = 5
REFERENCE_STAGE = 'retained'  # Paper specification; alternative: 'before_support'
assert REFERENCE_STAGE in {'retained','before_support'}
reference_inc = inc[inc.n_documents.ge(SUPPORT)] if REFERENCE_STAGE=='retained' else inc
reference_mass = reference_inc.groupby('task_id').weight.sum().reindex(task_ids,fill_value=0).to_numpy()
p = reference_mass/reference_mass.sum()
kept = inc[inc.n_documents.ge(SUPPORT)]
X = sparse.csr_matrix((kept.weight,(kept.code.map(ci),kept.task_id.map(ti))),
                      shape=(len(codes),len(task_ids)))
row_mass=np.asarray(X.sum(axis=1)).ravel()
q=sparse.diags(np.divide(1,row_mass,out=np.zeros_like(row_mass),where=row_mass>0))@X
inverse_p=np.divide(1,p,out=np.zeros_like(p),where=p>0)
rca=q.multiply(inverse_p).tocsr()
root=rca.copy();root.data=np.sqrt(root.data)
length=np.sqrt(np.asarray(root.multiply(root).sum(axis=1)).ravel())
v=(sparse.diags(np.divide(1,length,out=np.zeros_like(length),where=length>0))@root).tocsr()
nonempty=row_mass>0
np.testing.assert_allclose(np.asarray(v.multiply(v).sum(axis=1)).ravel()[nonempty],1,atol=1e-12)
assert not np.any((reference_mass==0)&(np.asarray(X.sum(axis=0)).ravel()>0))
pd.DataFrame({'task_id':task_ids,'reference_incidence':reference_mass,'corpus_task_share':p}).to_csv(OUT/'rca_reference.csv',index=False)
print('PASS: all nonempty vectors have unit length.')

Excluded links: 9083


PASS: all nonempty vectors have unit length.


## Work the example task by task

The contribution column sums to the RN–manager relatedness score. Tasks appearing in only one profile contribute zero, but still affect that profile's normalization. The two displayed profiles are detailed `.00` occupations, not pooled nursing specialties.

In [4]:
ORIGIN='29-1141.00'; DESTINATION='11-9111.00'
a,b=ci[ORIGIN],ci[DESTINATION]
score=float(v[a].multiply(v[b]).sum())
example=pd.DataFrame({'task_id':task_ids,'corpus_share':p,
 'rn_incidence':X[a].toarray().ravel(),'manager_incidence':X[b].toarray().ravel(),
 'rn_rca':rca[a].toarray().ravel(),'manager_rca':rca[b].toarray().ravel(),
 'rn_weight':v[a].toarray().ravel(),'manager_weight':v[b].toarray().ravel()})
example['task_text']=example.task_id.map(tasks.set_index('task_id').task_text)
example['contribution']=example.rn_weight*example.manager_weight
example=example[(example.rn_incidence>0)|(example.manager_incidence>0)].sort_values(['contribution','task_id'],ascending=[False,True])
np.testing.assert_allclose(example.contribution.sum(),score,atol=1e-12)
print(titles[ORIGIN], '→',titles[DESTINATION], '\nSimilarity:',score)
print('Shared retained tasks:',int(((example.rn_incidence>0)&(example.manager_incidence>0)).sum()))
display(example[['task_id','task_text','rn_rca','manager_rca','rn_weight','manager_weight','contribution']].head(20))
example.to_csv(OUT/'rn_manager_task_contributions.csv',index=False)

Registered Nurses → Medical and Health Services Managers 
Similarity: 0.4872831024461931
Shared retained tasks: 1643


,task_id,task_text,rn_rca,manager_rca,rn_weight,manager_weight,contribution
8353,10983,Collaborate with social service and health care professionals to advise individuals and families...,21.376537,35.326462,0.048691,0.078587,0.003826
5891,7782,"Plan, implement, or administer health programs or standards in hospitals, businesses, or communi...",14.759990,47.429046,0.040459,0.091059,0.003684
13195,17100,Direct the operations of short stay or specialty units.,13.624606,45.969691,0.038872,0.089647,0.003485
14251,18422,Monitor updates from public health agencies to keep abreast of health trends.,10.958174,50.123878,0.034861,0.093610,0.003263
1335,1864,Consult with institutions or associations regarding issues or concerns relevant to the practice ...,30.995978,17.074457,0.058631,0.054635,0.003203
12370,16035,Maintain knowledge of current events and trends in such areas as money laundering and criminal t...,21.058351,22.256587,0.048327,0.062377,0.003015
14784,19040,Collect information from individuals to compile vital statistics about the general health of com...,53.378593,8.186383,0.076941,0.037831,0.002911
13944,18013,"Instruct nursing staff in areas such as the assessment, development, implementation, and evaluat...",35.238509,11.621358,0.062515,0.045074,0.002818
15878,20416,Design evaluation programs regarding the quality and effectiveness of nursing practice or organi...,12.568731,31.087946,0.037335,0.073722,0.002752
13507,17481,"Monitor fetal growth and well-being through heartbeat detection, body measurement, and palpation.",38.896914,10.043798,0.065680,0.041903,0.002752


## Rank occupations and check the calculation

Rank positive scores, exclude the origin, and resolve ties with occupation-code order after rounding to 12 decimals, matching the existing pipeline. Here destinations must have at least 200 postings and a nonempty profile. This changes eligibility, not the RCA reference.

In [5]:
scores=(v@v.T).toarray()
np.testing.assert_allclose(scores,scores.T,atol=1e-12)
np.testing.assert_allclose(np.diag(scores)[nonempty],1,atol=1e-12)
assert scores.min()>=-1e-12 and scores.max()<=1+1e-12
n=totals.set_index('code').n_documents.reindex(codes).to_numpy()
eligible=(n>=200)&nonempty
j=np.flatnonzero(eligible&(np.arange(len(codes))!=a)&(scores[a]>0))
j=j[np.argsort(-np.round(scores[a,j],12),kind='stable')]
rankings=pd.DataFrame({'occupation':[codes[k] for k in j],
 'title':[titles[codes[k]] for k in j],'similarity':scores[a,j],'postings':n[j]})
rankings.insert(0,'rank',np.arange(1,len(rankings)+1))
display(rankings.head(20));rankings.to_csv(OUT/'rn_related_occupations.csv',index=False)
# The earlier analysis averages detailed-occupation pair scores at SOC6.
# It does NOT pool task counts and recompute a six-digit cosine.
rn_indices=[i for i,c in enumerate(codes) if c[:7]=='29-1141']
manager_indices=[i for i,c in enumerate(codes) if c[:7]=='11-9111']
soc6_score=float(scores[np.ix_(rn_indices,manager_indices)].mean())
print('Six-digit pair-mean score:',soc6_score)
np.savez_compressed(OUT/'task_similarity.npz',codes=np.array(codes),similarity=scores)
sparse.save_npz(OUT/'task_vectors.npz',v)
(OUT/'similarity_run.json').write_text(json.dumps({'review_rule':'NLx production pair exclusions; unreviewed links retained','exclusions_sha256':hashlib.sha256((DATA/'occupation_task_exclusions.csv').read_bytes()).hexdigest(),'support':SUPPORT,'reference_stage':REFERENCE_STAGE,'reference_occupations':len(codes),
 'rn_manager_detailed_score':score,'rn_manager_soc6_pair_mean':soc6_score,
 'shared_tasks':int(((example.rn_incidence>0)&(example.manager_incidence>0)).sum())},indent=2))

,rank,occupation,title,similarity,postings
0,1,29-1141.03,Critical Care Nurses,0.541926,5268
1,2,29-1171.00,Nurse Practitioners,0.522143,13395
2,3,29-1141.04,Clinical Nurse Specialists,0.518341,6989
3,4,29-2061.00,Licensed Practical and Licensed Vocational Nurses,0.488802,24739
4,5,11-9111.00,Medical and Health Services Managers,0.487283,32506
5,6,29-1141.02,Advanced Practice Psychiatric Nurses,0.485146,6403
6,7,29-1141.01,Acute Care Nurses,0.445070,2764
7,8,25-1072.00,"Nursing Instructors and Teachers, Postsecondary",0.414093,7796
8,9,31-1131.00,Nursing Assistants,0.385051,13356
9,10,15-1211.01,Health Informatics Specialists,0.365813,7363


Six-digit pair-mean score: 0.37713186041846386


379

## What these scores establish

These notebooks demonstrate the construction and interpretation of a task-based relatedness measure. They do not test observed transitions, establish individual eligibility for a job, or reproduce the paper's NLx estimates. Licensing, credentials and other requirements require separate evidence. The existing USAJOBS RCA specification was investigated retrospectively; it is not a preregistered or held-out model selection result.

Website/notebook explanatory text and code prepared with Codex; outputs are calculated from saved JAAT retrieval results. The documentation identifies extraction provenance, corpus access, and the support and reference populations.